In [69]:
import pandas as pd
import numpy as np
import sys
import os
import re
import json
import time
from tqdm import tqdm
from typing import List, Dict, Optional,Tuple, Any
from dataclasses import dataclass, field
import logging
from dotenv import load_dotenv
from pathlib import Path
import glob
import uuid as uuid_lib
# Корень проекта Executive_Exocortex
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv()

from pydantic import BaseModel, Field, field_validator

from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import entropy
from scipy.special import kl_div
from scipy import stats

from sentence_transformers import SentenceTransformer
from langchain_openai import ChatOpenAI
from openai import OpenAI
from langfuse import Langfuse

# atomizer, linker, GraphRAG
from config.settings import settings
from zettelkasten.atomizer import NoteAtomizer
from zettelkasten.linker import (
    GraphLinker,
    LocalEmbeddingModel,
    LinkAction,
)

import matplotlib.pyplot as plt
import seaborn as sns

In [134]:
# инициализация эмбеддинговой модели

embedding_model = LocalEmbeddingModel(
    model_name=settings.embedding_model_name,
)
EMBED_MODEL = SentenceTransformer("intfloat/multilingual-e5-base")

jugje_model = 'openai/gpt-5.1'
data_path = 'synthetic_datasets'
os.listdir('synthetic_datasets/')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9030.55it/s]


[EmbeddingModel] intfloat/multilingual-e5-base загружена за 13.7с (dim=768, device=mps)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11653.77it/s]


['linker', 'notes_dataset_v1.xlsx', 'atomizer']

# atomizer

```

┌──────────────────┬──────────────────────────────┬──────────────────────┐
│ Группа           │ Метрика                      │ Метод расчёта        │
├──────────────────┼──────────────────────────────┼──────────────────────┤
│ Производительность│ latency_mean_ms             │ среднее по заметкам  │
│                  │ latency_p95_ms               │ 95-й перцентиль      │
│                  │ cards_per_note_mean          │ среднее кол-во мыслей│
├──────────────────┼──────────────────────────────┼──────────────────────┤
│ Декомпозиция     │ count_ratio                  │ pred/ref количество  │
│                  │ count_mae                    │ |pred - ref| мыслей  │
├──────────────────┼──────────────────────────────┼──────────────────────┤
│ Иерархия         │ hierarchy_valid_ratio        │ корректность Луман-ID│
│                  │ root_count_match             │ кол-во корней совпало│
│                  │ depth_mae                    │ разница глубины дерева│
├──────────────────┼──────────────────────────────┼──────────────────────┤
│ Теги             │ tag_recall                   │ найденные эталон-теги│
│                  │ tag_precision                │ точность тегов       │
│                  │ tag_f1                       │ F1 по тегам          │
├──────────────────┼──────────────────────────────┼──────────────────────┤
│ Смысл            │ semantic_similarity_mean     │ cos sim эмбеддингов  │
│                  │ best_match_similarity        │ лучшее совпадение    │
│                  │ coverage_ratio               │ % покрытых этал мысл │
├──────────────────┼──────────────────────────────┼──────────────────────┤
│ Качество (LLM)   │ hallucination_score          │ LLM-as-judge 0-1     │
│                  │ faithfulness_score           │ соответствие тексту  │
│                  │ thought_type_accuracy        │ точность типа мысли  │
└──────────────────┴──────────────────────────────┴──────────────────────┘

```

| Метрика | Суть метрики | Формула в Markdown (LaTeX) |
| :--- | :--- | :--- |
| **Count Ratio** | Отношение количества предсказанных мыслей к эталонным (избыточность/недостаточность дробления) | $$Ratio = \frac{Count_{pred}}{Count_{ref}}$$ |
| **Count MAE** | Средняя абсолютная ошибка в количестве сгенерированных карточек | $$MAE = \|Count_{pred} - Count_{ref}\|$$ |
| **Depth MAE** | Абсолютная разница максимальной глубины дерева иерархии Лумана | $$Depth_{MAE} = \|Depth_{ref} - Depth_{pred}\|$$ |
| **Tag Precision** | Точность тегов: доля правильных тегов среди всех предсказанных моделью | $$Precision = \frac{\|Pred \cap Ref\|}{\|Pred\|}$$ |
| **Tag Recall** | Полнота тегов: доля найденных тегов среди всех эталонных в датасете | $$Recall = \frac{\|Pred \cap Ref\|}{\|Ref\|}$$ |
| **Tag F1-Score** | Гармоническое среднее точности и полноты для комплексной оценки тегов | $$F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}$$ |
| **Semantic Coverage Ratio** | Доля эталонных мыслей, покрытых предсказанием с косинусным сходством $\ge \tau$ | $$Coverage = \frac{1}{N_{ref}} \sum_{i=1}^{N_{ref}} \mathbb{I}(\max_{j} Sim(Ref_i, Pred_j) \ge \tau)$$ |
| **Hallucination Ratio** | Доля предсказанных мыслей, не нашедших семантического подтверждения в эталоне | $$Hallucination = \frac{1}{N_{pred}} \sum_{j=1}^{N_{pred}} \mathbb{I}(\max_{i} Sim(Pred_j, Ref_i) < \tau)$$ |
| **Distribution Overlap** | Степень пересечения распределений вероятностей (частоты) типов мыслей | $$Overlap = \sum_{t \in Types} \min(P_{ref}(t), P_{pred}(t))$$ |

`reference_thoughts_v1.xlsx` - правильные мысли на основе заметок, с ними будет сравниваться результат работы других моделей


In [ ]:
df_ref = pd.read_excel('synthetic_datasets/atomizer/reference_thoughts_v1.xlsx')
df_ref.head(3)

,note_id,note_text,zettel_id,luhmann_id,parent_id,parent_luhmann_id,content,thought_type,tags,parent_hint,is_root_topic,created_at,embedding
0,c09941ef-4576-4710-936d-dc12911f0ab7,Думаю по реструктуризации после разговора с Ма...,c2a4e8a7-82ac-4104-97a0-2847885ffef2,1,NaN,NaN,После разговора с Мариной и Thomas рассматрива...,context,"марина, thomas",NaN,True,2026-06-27 23:45:34.401,NaN
1,c09941ef-4576-4710-936d-dc12911f0ab7,Думаю по реструктуризации после разговора с Ма...,034b8012-a68e-473a-b109-b6bdc24bb1f6,1.1,c2a4e8a7-82ac-4104-97a0-2847885ffef2,1,Сокращение уровней в блоке «Операции» даст эко...,fact,операции,После разговора с Мариной и Thomas рассматрива...,False,2026-06-27 23:45:34.401,NaN
2,c09941ef-4576-4710-936d-dc12911f0ab7,Думаю по реструктуризации после разговора с Ма...,ddd2bdad-cd05-4816-a0c8-1c72abbaa0b5,1.1a,034b8012-a68e-473a-b109-b6bdc24bb1f6,1.1,Сокращение уровней в блоке «Операции» несет ри...,risk,"операции, сбермаркет",Сокращение уровней в блоке «Операции» даст эко...,False,2026-06-27 23:45:34.401,NaN


In [115]:
@dataclass
class NoteMetrics:
    """Все метрики для одной заметки"""
    note_id: str
    model_name: str
    
    # Производительность
    latency_ms: float = 0.0
    
    # Декомпозиция
    ref_count: int = 0         # сколько мыслей в эталоне
    pred_count: int = 0        # сколько мыслей у модели
    count_ratio: float = 0.0   # pred / ref
    count_mae: float = 0.0     # |pred - ref|
    
    # Иерархия
    hierarchy_valid_ratio: float = 0.0  # доля валидных Луман-ID
    root_count_ref: int = 0
    root_count_pred: int = 0
    depth_mae: float = 0.0
    parent_consistency: float = 0.0     # % карточек с правильным parent_hint
    
    # Теги
    tag_precision: float = 0.0
    tag_recall: float = 0.0
    tag_f1: float = 0.0
    
    # Смысл (семантика)
    semantic_similarity_mean: float = 0.0
    semantic_coverage: float = 0.0      # % эталонных мыслей покрытых предсказанием
    
    # Типы мыслей
    thought_type_accuracy: float = 0.0
    
    # LLM-as-judge
    hallucination_score: float = 0.0
    faithfulness_score: float = 0.0


## метрики декомпозиции заметки на атомарные мысли

In [116]:
#  метрики декомпозиции заметки на мысли

def compute_count_metrics(ref_cards: List[dict], pred_cards: List[dict]) -> dict:
    """
    Считает метрики количества мыслей.
    
    ref_cards  — эталонные карточки (из датасета Langfuse)
    pred_cards — карточки от тестируемой модели
    """
    ref_count = len(ref_cards)
    pred_count = len(pred_cards)
    
    count_ratio = pred_count / ref_count if ref_count > 0 else 0.0
    count_mae = abs(pred_count - ref_count)
    
    return {
        "ref_count": ref_count,
        "pred_count": pred_count,
        "count_ratio": round(count_ratio, 3),
        "count_mae": count_mae,
        # Интерпретация: 1.0 = идеально, >1.0 = модель дробит лишнее, <1.0 = недодробила
    }


## метрики иерархии (по luhmann_id)

In [117]:
# метрики иерархии (по Луман-ID)
def parse_luhmann_depth(luhmann_id: str) -> Optional[int]:
    """
    Возвращает глубину карточки по Луман-ID.
    
    Примеры:
        "1"     → глубина 1 (корень)
        "1.1"   → глубина 2
        "1.1a"  → глубина 3
        "1.1a1" → глубина 4
    """
    if not luhmann_id or not isinstance(luhmann_id, str):
        return None
    
    # Считаем количество уровней: цифры и буквы чередуются
    # Паттерн Лумана: число (.число | буква)*
    pattern = r'^\d+([a-z]\d*|\.\d+[a-z]?)*$'
    if not re.match(pattern, luhmann_id.strip(), re.IGNORECASE):
        return None  # невалидный ID
    
    # Считаем переходы: каждая точка или смена цифра→буква = новый уровень
    depth = 1
    prev_was_digit = True
    for ch in luhmann_id[1:]:  # пропускаем первую цифру
        if ch == '.':
            depth += 1
            prev_was_digit = True
        elif ch.isalpha() and prev_was_digit:
            depth += 1
            prev_was_digit = False
        elif ch.isdigit() and not prev_was_digit:
            depth += 1
            prev_was_digit = True
    
    return depth


def validate_luhmann_ids(cards: List[dict]) -> dict:
    """
    Проверяет корректность структуры Луман-ID внутри одной заметки.
    
    Проверяет:
    1. Все ID уникальны
    2. Все ID имеют правильный формат
    3. Для каждого дочернего ID существует родительский
    """
    ids = [card.get('luhmann_id', '') for card in cards]
    valid_ids = [lid for lid in ids if parse_luhmann_depth(lid) is not None]
    
    # Проверка уникальности
    unique_ratio = len(set(ids)) / len(ids) if ids else 0
    
    # Проверка формата
    format_valid_ratio = len(valid_ids) / len(ids) if ids else 0
    
    # Проверка родительских связей: у "1.1" должен быть "1"
    id_set = set(ids)
    parent_exists_count = 0
    child_count = 0
    
    for lid in valid_ids:
        depth = parse_luhmann_depth(lid)
        if depth and depth > 1:
            child_count += 1
            # Определяем ожидаемого родителя
            parent_id = get_expected_parent(lid)
            if parent_id in id_set:
                parent_exists_count += 1
    
    parent_consistency = (
        parent_exists_count / child_count 
        if child_count > 0 else 1.0  # нет дочерних = всё корни = окей
    )
    
    depths = [parse_luhmann_depth(lid) for lid in valid_ids]
    depths = [d for d in depths if d is not None]
    
    return {
        "hierarchy_valid_ratio": round(format_valid_ratio, 3),
        "unique_ratio": round(unique_ratio, 3),
        "parent_consistency": round(parent_consistency, 3),
        "max_depth": max(depths) if depths else 0,
        "mean_depth": round(np.mean(depths), 2) if depths else 0,
    }


def get_expected_parent(luhmann_id: str) -> Optional[str]:
    """
    По Луман-ID дочернего узла вычисляет ID ожидаемого родителя.
    
    Примеры:
        "1.1"  → "1"
        "1.1a" → "1.1"
        "1.1a1"→ "1.1a"
    """
    lid = luhmann_id.strip()
    
    # Ищем последнее "разделение" — точку или переход цифра-буква/буква-цифра
    # Идём с конца
    for i in range(len(lid) - 1, 0, -1):
        if lid[i] == '.':
            return lid[:i]
        elif lid[i].isalpha() and lid[i-1].isdigit():
            return lid[:i]
        elif lid[i].isdigit() and lid[i-1].isalpha():
            return lid[:i]
    
    return None


def compute_hierarchy_metrics(ref_cards: List[dict], pred_cards: List[dict]) -> dict:
    """Сравнивает иерархическую структуру предсказания с эталоном"""
    
    # Валидность структур
    ref_hierarchy = validate_luhmann_ids(ref_cards)
    pred_hierarchy = validate_luhmann_ids(pred_cards)
    
    # Количество корневых мыслей
    ref_roots = sum(1 for c in ref_cards if c.get('is_root_topic', False))
    pred_roots = sum(1 for c in pred_cards if c.get('is_root_topic', False))
    
    # Разница в глубине дерева
    depth_mae = abs(ref_hierarchy['max_depth'] - pred_hierarchy['max_depth'])
    mean_depth_mae = abs(ref_hierarchy['mean_depth'] - pred_hierarchy['mean_depth'])
    
    return {
        # Качество структуры предсказания (абсолютное)
        "pred_hierarchy_valid_ratio": pred_hierarchy['hierarchy_valid_ratio'],
        "pred_parent_consistency": pred_hierarchy['parent_consistency'],
        "pred_max_depth": pred_hierarchy['max_depth'],
        
        # Сравнение с эталоном
        "root_count_ref": ref_roots,
        "root_count_pred": pred_roots,
        "root_count_delta": abs(ref_roots - pred_roots),
        "depth_mae": depth_mae,
        "mean_depth_mae": round(mean_depth_mae, 2),
    }


## метрики корректности извлечения тегов

In [118]:
# метрики корректности выделения тегов
def normalize_tag(tag: str) -> str:
    """Нормализует тег: нижний регистр, snake_case, убираем пробелы"""
    tag = tag.strip().lower()
    tag = re.sub(r'\s+', '_', tag)
    tag = re.sub(r'[^\w_]', '', tag)
    return tag


def compute_tag_metrics(ref_cards: List[dict], pred_cards: List[dict]) -> dict:
    """
    Считает precision/recall/F1 по тегам на уровне всей заметки.
    
    Подход: берём ВСЕ теги из всех карточек заметки и сравниваем множества.
    """
    # Собираем все теги из эталона
    ref_tags = set()
    for card in ref_cards:
        tags_raw = card.get('tags', '')
        if isinstance(tags_raw, str):
            for t in tags_raw.split(','):
                normalized = normalize_tag(t)
                if normalized:
                    ref_tags.add(normalized)
        elif isinstance(tags_raw, list):
            for t in tags_raw:
                normalized = normalize_tag(str(t))
                if normalized:
                    ref_tags.add(normalized)
    
    # Собираем все теги из предсказания
    pred_tags = set()
    for card in pred_cards:
        tags_raw = card.get('tags', '')
        if isinstance(tags_raw, str):
            for t in tags_raw.split(','):
                normalized = normalize_tag(t)
                if normalized:
                    pred_tags.add(normalized)
        elif isinstance(tags_raw, list):
            for t in tags_raw:
                normalized = normalize_tag(str(t))
                if normalized:
                    pred_tags.add(normalized)
    
    # Считаем метрики
    if not ref_tags and not pred_tags:
        return {"tag_precision": 1.0, "tag_recall": 1.0, "tag_f1": 1.0,
                "ref_tag_count": 0, "pred_tag_count": 0}
    
    intersection = ref_tags & pred_tags
    
    precision = len(intersection) / len(pred_tags) if pred_tags else 0.0
    recall = len(intersection) / len(ref_tags) if ref_tags else 0.0
    f1 = (2 * precision * recall / (precision + recall) 
          if (precision + recall) > 0 else 0.0)
    
    return {
        "tag_precision": round(precision, 3),
        "tag_recall": round(recall, 3),
        "tag_f1": round(f1, 3),
        "ref_tag_count": len(ref_tags),
        "pred_tag_count": len(pred_tags),
        "missed_tags": list(ref_tags - pred_tags),     # что пропустила модель
        "extra_tags": list(pred_tags - ref_tags),       # что добавила лишнего
    }



## метрики оценки формулирования мыслей (семантические метрики)

In [119]:
# семантические метрики (смысл мыслей)
def get_embeddings(texts: List[str]) -> np.ndarray:
    """Получает эмбеддинги для списка текстов"""
    # prefix нужен для e5-моделей
    prefixed = [f"passage: {t}" for t in texts]
    return EMBED_MODEL.encode(prefixed, normalize_embeddings=True)


def compute_semantic_metrics(
    ref_cards: List[dict], 
    pred_cards: List[dict],
    coverage_threshold: float = 0.75  # минимальное сходство для "покрытия"
) -> dict:
    """
    Считает семантические метрики через косинусное сходство эмбеддингов.
    
    Алгоритм:
    1. Для каждой эталонной мысли находим наиболее похожую предсказанную
    2. Считаем среднее максимальное сходство (Coverage Mean Similarity)
    3. Считаем % эталонных мыслей, для которых нашлась пара (Coverage Ratio)
    """
    ref_texts = [card.get('content', '') for card in ref_cards]
    pred_texts = [card.get('content', '') for card in pred_cards]
    
    if not ref_texts or not pred_texts:
        return {
            "semantic_similarity_mean": 0.0,
            "semantic_coverage_ratio": 0.0,
            "semantic_similarity_std": 0.0,
        }
    
    # Получаем эмбеддинги
    ref_embeddings = get_embeddings(ref_texts)    # shape: (n_ref, dim)
    pred_embeddings = get_embeddings(pred_texts)  # shape: (n_pred, dim)
    
    # Матрица сходства: строки = эталон, столбцы = предсказание
    sim_matrix = cosine_similarity(ref_embeddings, pred_embeddings)
    # shape: (n_ref, n_pred)
    
    # Для каждой эталонной мысли — максимальное сходство с любой предсказанной
    best_matches = sim_matrix.max(axis=1)  # shape: (n_ref,)
    
    # Средняя похожесть лучших матчей
    similarity_mean = float(np.mean(best_matches))
    similarity_std = float(np.std(best_matches))
    
    # Процент покрытых эталонных мыслей
    covered = np.sum(best_matches >= coverage_threshold)
    coverage_ratio = covered / len(ref_texts)
    
    # Дополнительно: симметричная метрика — насколько предсказанные мысли
    # обоснованы в эталоне (анти-галлюцинация)
    pred_best_matches = sim_matrix.max(axis=0)  # для каждой pred-мысли
    hallucination_candidates = np.sum(pred_best_matches < coverage_threshold)
    hallucination_ratio = hallucination_candidates / len(pred_texts)
    
    return {
        "semantic_similarity_mean": round(similarity_mean, 4),
        "semantic_similarity_std": round(similarity_std, 4),
        "semantic_coverage_ratio": round(float(coverage_ratio), 4),
        "hallucination_ratio": round(hallucination_ratio, 4),
        # Детали для отладки
        "worst_covered_ref": ref_texts[int(np.argmin(best_matches))],
        "best_matched_ref": ref_texts[int(np.argmax(best_matches))],
    }



## метрики оценки типов мыслей

In [120]:
# точность типов мыслей

def compute_thought_type_metrics(ref_cards: List[dict], pred_cards: List[dict]) -> dict:
    """
    Сравнивает распределение типов мыслей.
    
    Поскольку карточки не выровнены (нет взаимно-однозначного соответствия),
    сравниваем распределения типов, а не попарно.
    """
    from collections import Counter
    
    ref_types = Counter(card.get('thought_type', 'unknown') for card in ref_cards)
    pred_types = Counter(card.get('thought_type', 'unknown') for card in pred_cards)
    
    all_types = set(ref_types.keys()) | set(pred_types.keys())
    
    # Нормализуем в вероятности
    ref_total = sum(ref_types.values()) or 1
    pred_total = sum(pred_types.values()) or 1
    
    # Jensen-Shannon divergence между распределениями типов
    ref_probs = np.array([ref_types.get(t, 0) / ref_total for t in sorted(all_types)])
    pred_probs = np.array([pred_types.get(t, 0) / pred_total for t in sorted(all_types)])
    
    # Простая метрика: пересечение распределений
    overlap = sum(min(ref_types.get(t, 0)/ref_total, pred_types.get(t, 0)/pred_total) 
                  for t in all_types)
    
    return {
        "thought_type_distribution_overlap": round(overlap, 3),
        "ref_type_counts": dict(ref_types),
        "pred_type_counts": dict(pred_types),
        "type_count_delta": {
            t: pred_types.get(t, 0) - ref_types.get(t, 0) 
            for t in all_types
        }
    }

## LLM as a judge - проверка на галлюцинации

In [121]:
jugje_model = 'openai/gpt-5.1'

In [122]:
client = OpenAI(
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)


In [123]:
class AtomizerEvaluationSchema(BaseModel):
    """Строгая схема оценки качества атомизации по методологии G-Eval"""
    
    chain_of_thought: str = Field(
        ...,
        description="Пошаговые рассуждения (Chain of Thought). Проанализируйте исходную заметку "
                    "и извлеченные мысли. Выявите любые искажения смысла или придуманные факты."
    )
    faithfulness: float = Field(
        ...,
        description="Оценка верности источнику от 0.0 до 1.0. "
                    "1.0 - все мысли на 100% подтверждаются текстом. "
                    "0.5 - есть небольшие искажения контекста или домыслы. "
                    "0.0 - мысли полностью противоречат тексту или выдуманы."
    )
    hallucination_score: float = Field(
        ...,
        description="Показатель галлюцинаций от 0.0 до 1.0. "
                    "0.0 - галлюцинаций нет. "
                    "0.5 - добавлены факты, которых не было в тексте, но они выглядят правдоподобно. "
                    "1.0 - модель грубо придумала несуществующие сущности, связи, цифры или имена."
    )
    detected_errors: List[str] = Field(
        default_factory=list,
        description="Список конкретных нестыковок или галлюцинаций. Если ошибок нет, оставьте список пустым."
    )

    # Строгие валидаторы (Pedantic)
    @field_validator("faithfulness", "hallucination_score")
    @classmethod
    def validate_scores(cls, val: float) -> float:
        if not (0.0 <= val <= 1.0):
            raise ValueError("Оценка должна быть строго в диапазоне от 0.0 до 1.0")
        return round(val, 2)

    @field_validator("chain_of_thought")
    @classmethod
    def validate_reasoning(cls, val: str) -> str:
        if len(val.strip()) < 30:
            raise ValueError("Рассуждение (chain_of_thought) должно быть развернутым (минимум 30 символов)")
        return val.strip()

In [124]:
# g-eval
# Системный промпт по методологии G-Eval

GEVAL_SYSTEM_PROMPT = """Вы — эксперт-аудитор систем управления знаниями (Zettelkasten). 
Ваша задача — оценить качество декомпозиции (атомизации) исходного текста на отдельные карточки-мысли.

Методология оценки (G-Eval):
1. Внимательно прочитайте ИСХОДНЫЙ ТЕКСТ заметки.
2. Изучите СПИСОК ИЗВЛЕЧЕННЫХ МЫСЛЕЙ (выход тестируемой модели).
3. Проведите пошаговый анализ (Chain of Thought):
   а. Каждая ли извлеченная мысль прямо подтверждается фактами из исходного текста?
   б. Нет ли добавления внешней информации, которой не было в источнику (галлюцинации)?
   в. Не искажен ли смысл при перефразировании?
4. Сформируйте список конкретных ошибок (если они есть).
5. Выставьте оценки:
   - Faithfulness (верность): 1.0 (идеально), 0.0 (полная отсебятина).
   - Hallucination Score (галлюцинации): 0.0 (нет галлюцинаций), 1.0 (грубые выдумки)."""

USER_PROMPT_TEMPLATE = """
--- ИСХОДНЫЙ ТЕКСТ ЗАМЕТКИ ---
{note_text}

--- ИЗВЛЕЧЕННЫЕ МЫСЛИ (Тестируемая модель) ---
{pred_thoughts}
"""

In [125]:
def judge_atomization(
    note_text: str, 
    pred_cards: List[dict],
    model: str = jugje_model 
) -> dict:
    """
    LLM-судья на базе методологии G-Eval и Pydantic-валидации.
    Возвращает словарь с ключами, совместимыми с основным пайплайном оценки.
    """
    # Если на входе нет карточек, сразу штрафуем модель
    if not pred_cards:
        return {
            "faithfulness_score": 0.0,
            "hallucination_score": 1.0,
            "judge_comment": "Ошибка: Модель вернула пустой результат."
        }

    # Форматируем мысли для судьи
    pred_thoughts_str = "\n".join([
        f"- Luhmann ID: {card.get('luhmann_id', 'N/A')} | "
        f"Type: {card.get('thought_type', 'unknown')} | "
        f"Content: {card.get('content', '')}"
        for card in pred_cards
    ])

    try:
        # Используем Structured Outputs API (нативная поддержка Pydantic)
        completion = client.beta.chat.completions.parse(
            model=jugje_model,
            messages=[
                {"role": "system", "content": GEVAL_SYSTEM_PROMPT},
                {"role": "user", "content": USER_PROMPT_TEMPLATE.format(
                    note_text=note_text,
                    pred_thoughts=pred_thoughts_str
                )}
            ],
            response_format=AtomizerEvaluationSchema,
            temperature=0.0, # строгость оценки
            timeout=60,
        )

        # Вытаскиваем распарсенный Pydantic объект
        result: AtomizerEvaluationSchema = completion.choices[0].message.parsed
        
        # Формируем красивый комментарий для Langfuse/Excel
        errors_str = f" | Ошибки: {'; '.join(result.detected_errors)}" if result.detected_errors else ""
        comment = f"[CoT]: {result.chain_of_thought}{errors_str}"

        # Возвращаем в формате, который ожидает ваш скрипт
        return {
            "faithfulness_score": result.faithfulness,
            "hallucination_score": result.hallucination_score,
            "judge_comment": comment
        }

    except Exception as e:
        print(f"ошибка LLM-судьи: {e}")
        # В случае ошибки API возвращаем нейтральные пессимистичные метрики, чтобы не падал скрипт
        return {
            "faithfulness_score": 0.5,
            "hallucination_score": 0.5,
            "judge_comment": f"Ошибка вызова LLM-судьи: {str(e)}"
        }

In [ ]:
def load_reference_data(path: str = "synthetic_datasets/atomizer/reference_thoughts_v1.xlsx") -> dict:
    """
    Загружает эталонные данные и группирует по note_id.
    Возвращает dict: {note_id: [список карточек]}
    """
    df = pd.read_excel(path)
    reference = {}
    
    for note_id, group in df.groupby('note_id'):
        reference[note_id] = group.to_dict('records')
    
    return reference

## main eval pipeline 

In [127]:
def evaluate_model_file(
    model_file: str,
    reference: dict,
    run_llm_judge: bool = True,
    judge_sample_size: int = 50,  # оцениваем только N заметок через LLM (дорого)
) -> pd.DataFrame:
    """
    Считает все метрики для одного файла результатов модели.
    """
    df_pred = pd.read_excel(model_file)
    model_name = df_pred['model_name'].iloc[0]
    
    results = []
    
    # Группируем предсказания по заметкам
    pred_by_note = {}
    for note_id, group in df_pred.groupby('note_id'):
        pred_by_note[note_id] = group.to_dict('records')
    
    # Для LLM-judge берём случайную выборку заметок
    import random
    note_ids_for_judge = set(
        random.sample(list(reference.keys()), 
                      min(judge_sample_size, len(reference)))
    )
    
    for note_id, ref_cards in tqdm(reference.items(), desc=f"Оценка {model_name}"):
        pred_cards = pred_by_note.get(note_id, [])
        
        if not pred_cards:
            # Модель не обработала эту заметку — штраф
            results.append({
                "note_id": note_id,
                "model_name": model_name,
                "error": "no_output",
                **{k: 0.0 for k in [
                    "latency_ms_total", "count_ratio", "count_mae",
                    "pred_hierarchy_valid_ratio", "root_count_delta",
                    "tag_f1", "tag_precision", "tag_recall",
                    "semantic_similarity_mean", "semantic_coverage_ratio",
                    "hallucination_ratio", "faithfulness_score",
                    "hallucination_score", "thought_type_distribution_overlap"
                ]}
            })
            continue
        
        # Берём латентность (одинакова для всех карточек одной заметки)
        latency_ms = pred_cards[0].get('latency_ms_total', 0.0)
        note_text = pred_cards[0].get('note_text', '')
        
        # ── Считаем все метрики ──────────────────────────────────
        count_m = compute_count_metrics(ref_cards, pred_cards)
        hierarchy_m = compute_hierarchy_metrics(ref_cards, pred_cards)
        tag_m = compute_tag_metrics(ref_cards, pred_cards)
        semantic_m = compute_semantic_metrics(ref_cards, pred_cards)
        type_m = compute_thought_type_metrics(ref_cards, pred_cards)
        
        # LLM-judge (только для выборки)
        judge_m = {"faithfulness_score": None, "hallucination_score": None}
        if run_llm_judge and note_id in note_ids_for_judge:
            judge_m = judge_atomization(note_text, pred_cards)
        
        # ── Собираем результат ───────────────────────────────────
        row = {
            "note_id": note_id,
            "model_name": model_name,
            "latency_ms_total": latency_ms,
            **count_m,
            **hierarchy_m,
            **tag_m,
            **{k: v for k, v in semantic_m.items() 
               if not isinstance(v, str)},  # исключаем текстовые детали
            **{k: v for k, v in type_m.items() 
               if not isinstance(v, dict)},  # исключаем сложные объекты
            **judge_m,
        }
        results.append(row)
    
    return pd.DataFrame(results)


def compute_summary(df_metrics: pd.DataFrame) -> pd.DataFrame:
    """Агрегирует метрики по модели для итогового сравнения"""
    
    numeric_cols = df_metrics.select_dtypes(include=[np.number]).columns.tolist()
    
    summary = df_metrics.groupby('model_name')[numeric_cols].agg([
        'mean', 'median', 'std',
        lambda x: x.quantile(0.05),   # P5
        lambda x: x.quantile(0.95),   # P95
    ]).round(4)
    
    return summary



In [131]:
def evaluate_atomizer():
    reference = load_reference_data()
    
    model_files = list(Path("synthetic_datasets").glob("thoughts_*.xlsx"))
    
    all_metrics = []
    
    for model_file in model_files:
        print(f"\n -->Оцениваем: {model_file.name}")
        df_metrics = evaluate_model_file(
            str(model_file), 
            reference,
            run_llm_judge=True,
            judge_sample_size=50, # оцениваем только N заметок через LLM 
        )
        all_metrics.append(df_metrics)
        
        # Сохраняем детальные метрики для каждой модели
        df_metrics.to_excel(
            f"metric_results/atomizer/atomizer_metrics_{model_file.stem}.xlsx", 
            index=False
        )
    
    # Объединяем всё вместе
    df_all = pd.concat(all_metrics, ignore_index=True)
    df_all.to_excel("metric_results/atomizer/all_models_atomizer_metrics.xlsx", index=False)
    
    # Итоговая таблица сравнения
    summary = compute_summary(df_all)
    print()
    print("[Сравнение моделей по атомизации]")
    print()
    print(summary.to_string())
    
    summary.to_excel("metric_results/atomizer/summary_comparison.xlsx")

In [132]:
evaluate_atomizer()


 -->Оцениваем: thoughts_anthropic_claude-haiku-4.5.xlsx


Оценка anthropic/claude-haiku-4.5: 100%|██████████| 200/200 [11:29<00:00,  3.45s/it]



 -->Оцениваем: thoughts_openai_gpt-4o-mini.xlsx


Оценка openai/gpt-4o-mini: 100%|██████████| 200/200 [09:12<00:00,  2.76s/it]



 -->Оцениваем: thoughts_google_gemini-2.5-flash.xlsx


...

ошибка LLM-судьи: Error code: 503 - {'error': {'message': 'litellm.ServiceUnavailableError: ServiceUnavailableError: OpenrouterException - <html><body><h1>503 Service Unavailable</h1>\nNo server is available to handle this request.\n</body></html>\n. Received Model Group=openai/gpt-5.1\nAvailable Model Group Fallbacks=None', 'type': None, 'param': None, 'code': '503'}}


Оценка google/gemini-2.5-flash:  37%|███▋      | 74/200 [02:36<02:19,  1.11s/it]

ошибка LLM-судьи: Error code: 503 - {'error': {'message': 'litellm.ServiceUnavailableError: ServiceUnavailableError: OpenrouterException - <html><body><h1>503 Service Unavailable</h1>\nNo server is available to handle this request.\n</body></html>\n. Received Model Group=openai/gpt-5.1\nAvailable Model Group Fallbacks=None', 'type': None, 'param': None, 'code': '503'}}


Оценка google/gemini-2.5-flash: 100%|██████████| 200/200 [08:21<00:00,  2.51s/it]



[Сравнение моделей по атомизации]

                           latency_ms_total                                             ref_count                                     pred_count                                      count_ratio                                      count_mae                                      pred_hierarchy_valid_ratio                                   pred_parent_consistency                                   pred_max_depth                                      root_count_ref                                      root_count_pred                                      root_count_delta                                      depth_mae                                      mean_depth_mae                                      tag_precision                                       tag_recall                                       tag_f1                                       ref_tag_count                                      pred_tag_count                                       semanti

# Linker


| Метрика | Что означает | Формула |
| :--- | :--- | :--- |
| **Action Accuracy** | Доля карточек, где модель выбрала то же действие, что и эталон | $$ACC = \frac{\sum(action_{test} == action_{ref})}{N}$$ |
| **Graph Depth Consistency** | Совпадение глубины встройки (число компонент Luhmann ID) с эталоном | $$GDC = 1 - \frac{1}{N}\sum \frac{|depth_{test} - depth_{ref}|}{max\_depth}$$ |
| **Root Rate Delta** | Отклонение доли `new_root` решений от эталона | $$RRD = | \frac{N_{root\_test}}{N_{test}} - \frac{N_{root\_ref}}{N_{ref}} |$$ |
| **Child Attachment Precision** | Точность выбора уровня узла для `child_of` (по Luhmann-префиксу) | $$CAP = \frac{\sum(luhmann\_prefix\_match)}{N_{child}}$$ |
| **Graph Structure Similarity** | Корреляция Спирмена распределения глубин узлов | $$GSS = SpearmanR(depth\_dist_{test}, depth\_dist_{ref})$$ |
| **Action Dist. KL-Divergence** | Расхождение распределения типов действий (чем ближе к 0, тем лучше) | $$KL(P\|Q) = \sum P(a) \cdot \log\frac{P(a)}{Q(a)}$$ |
| **Latency per Card** | Среднее время обработки одной карточки | $$LPC = \frac{total\_latency}{N}$$ |
| **Cost Efficiency Score** | Качество (Action Accuracy) на единицу стоимости модели | $$CES = \frac{ACC}{mean\_price}$$ |

In [36]:
LINKER_DIR   = Path("synthetic_datasets/linker")
REFERENCE_FILE = LINKER_DIR / "reference_linker_gpt-5.2.xlsx"
OUTPUT_FILE    =  "metric_results/linker/linker_metrics_summary.xlsx"


In [ ]:
import logging
from typing import Any, Tuple

from storage.neo4j.client import get_neo4j_client
from storage.neo4j.repository import ZettelRepository

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("retrieval_eval")

TOP_K_VALUES = [3, 5, 7]
TEST_USER_ID = "linker_eval_gpt-5.2"
QUESTIONS_PATH = Path("synthetic_datasets/synthetic_questions.xlsx")
NOTES_PATH = Path("synthetic_datasets/notes_dataset_v1.xlsx")
OUTPUT_DIR = Path("metric_results/rag/retrieval")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_QUESTION_COLUMNS = {
    "question_id",
    "question",
    "reference_answer",
    "note_id",
}


def _build_simple_questions_from_notes(df_notes: pd.DataFrame, n_questions: int = 200) -> pd.DataFrame:
    """Бэкап-генерация тестовых вопросов из notes_dataset_v1.xlsx."""
    if df_notes.empty:
        raise ValueError("notes_dataset_v1.xlsx пустой — нечего использовать для генерации вопросов")

    sample = df_notes.sample(n=min(n_questions, len(df_notes)), random_state=42).reset_index(drop=True)
    rows = []
    for i, row in sample.iterrows():
        note_text = str(row.get("note_text", "")).strip()
        note_id = str(row.get("note_id", "")).strip()
        if not note_text or not note_id:
            continue
        qid = f"Q-BACKUP-{i+1:04d}"
        rows.append(
            {
                "question_id": qid,
                "question": f"Кратко напомни ключевые мысли из заметки {qid}.",
                "reference_answer": note_text,
                "key_concepts": "",
                "reasoning": "backup_generation",
                "question_type": "summary",
                "question_style": "direct",
                "note_id": note_id,
                "note_text": note_text,
                "note_domain": row.get("domain", ""),
                "note_style": row.get("style", ""),
                "note_created_at": row.get("created_at", ""),
                "generated_at": pd.Timestamp.utcnow(),
                "model_name": "backup_rule_based",
            }
        )

    if not rows:
        raise ValueError("Не удалось собрать ни одного вопроса из notes_dataset_v1.xlsx")

    return pd.DataFrame(rows)


def ensure_questions_dataset(path: Path = QUESTIONS_PATH, n_questions: int = 200) -> pd.DataFrame:
    """
    Загружает synthetic_questions.xlsx. Если файл отсутствует/поврежден/нет нужных колонок,
    пытается перегенерировать датасет вопросов.
    """
    regenerate_needed = False

    if not path.exists():
        logger.warning("Файл %s не найден. Запускаем перегенерацию.", path)
        regenerate_needed = True

    if not regenerate_needed:
        try:
            df_q = pd.read_excel(path)
            missing = REQUIRED_QUESTION_COLUMNS - set(df_q.columns)
            if missing:
                logger.warning("В %s нет колонок %s. Нужна перегенерация.", path, sorted(missing))
                regenerate_needed = True
            elif df_q.empty:
                logger.warning("%s пустой. Нужна перегенерация.", path)
                regenerate_needed = True
            else:
                logger.info("Загружено вопросов: %d", len(df_q))
                logger.info("Колонки: %s", list(df_q.columns))
                return df_q
        except Exception as e:
            logger.warning("Не удалось прочитать %s (%s). Нужна перегенерация.", path, e)
            regenerate_needed = True

    if not NOTES_PATH.exists():
        raise FileNotFoundError(
            f"Нет исходного датасета для перегенерации: {NOTES_PATH}. "
            "Сначала сгенерируй заметки в generate_data.ipynb."
        )

    df_notes = pd.read_excel(NOTES_PATH)

    if "SyntheticQuestionGenerator" in globals() and "GENERATE_LLM_MODEL" in globals():
        logger.info("Найден SyntheticQuestionGenerator в окружении. Перегенерируем через LLM.")
        generator = SyntheticQuestionGenerator(model_name=GENERATE_LLM_MODEL, temperature=0.85)
        df_q = generator.generate_dataset(
            df_notes=df_notes,
            num_questions=n_questions,
            output_excel_path=str(path),
            save_every=20,
            max_workers=8,
        )
        if not isinstance(df_q, pd.DataFrame):
            df_q = pd.DataFrame(df_q)
    else:
        logger.warning(
            "SyntheticQuestionGenerator не найден. Используем backup-генерацию вопросов "
            "из notes_dataset_v1.xlsx."
        )
        df_q = _build_simple_questions_from_notes(df_notes, n_questions=n_questions)
        df_q.to_excel(path, index=False)

    missing = REQUIRED_QUESTION_COLUMNS - set(df_q.columns)
    if missing:
        raise ValueError(f"После перегенерации всё еще нет обязательных колонок: {sorted(missing)}")

    logger.info("Перегенерировано вопросов: %d. Сохранено в %s", len(df_q), path)
    return df_q

In [16]:
# стоимость моделей (₽ за 1М токенов)

MODEL_COSTS = {
    "gpt-4o-mini":        {"input": 16.2,  "output": 64.8,  "mean": 40.5},
    "gemini-2.5-flash":   {"input": 32.4,  "output": 270.0, "mean": 151.2},
    "claude-haiku-4.5":   {"input": 108.0, "output": 540.0, "mean": 324.0},
}

# Маппинг: часть имени файла → ключ в MODEL_COSTS
FILE_TO_MODEL_KEY = {
    "gpt-4o-mini":       "gpt-4o-mini",
    "gemini-2.5-flash":  "gemini-2.5-flash",
    "claude-haiku-4.5":  "claude-haiku-4.5",
}

## загрузка тестовых данных для линкера

In [25]:
def extract_model_key(filepath: str) -> str:
    """Извлекает читаемое название модели из имени файла."""
    name = Path(filepath).stem  # linker_gpt-4o-mini
    # Убираем префикс linker_
    name = name.replace("linker_", "").replace("openai_", "").replace("google_", "").replace("anthropic_", "")
    # Убираем дублирующиеся префиксы вендора (артефакт safe_model_name)
    for key in FILE_TO_MODEL_KEY:
        if key.replace("-", "_") in name.replace("-", "_"):
            return key
    return name


def load_reference(filepath: str | Path) -> pd.DataFrame:
    """Загружает эталонный датафрейм."""
    df = pd.read_excel(filepath)
    print(f"[Reference] Загружено {len(df)} карточек из {filepath}")
    return df


def load_test_files(directory: str | Path, exclude: str = "reference") -> dict[str, pd.DataFrame]:
    """
    Загружает все файлы тестируемых моделей из директории.
    Пропускает файлы с 'reference' в названии.
    
    Returns: словарь {model_key: DataFrame}
    """
    pattern = str(Path(directory) / "linker_*.xlsx")
    files = glob.glob(pattern)
    
    result = {}
    for fp in sorted(files):
        if exclude in Path(fp).name:
            continue
        model_key = extract_model_key(fp)
        df = pd.read_excel(fp)
        print(f"[Test] {model_key}: {len(df)} карточек из {Path(fp).name}")
        result[model_key] = df
    
    return result


def add_cost_columns(metrics_df: pd.DataFrame) -> pd.DataFrame:
    """Добавляет столбцы со стоимостью модели."""
    metrics_df["cost_input_per_M"]  = metrics_df["model_name"].map(
        lambda m: MODEL_COSTS.get(m, {}).get("input", None)
    )
    metrics_df["cost_output_per_M"] = metrics_df["model_name"].map(
        lambda m: MODEL_COSTS.get(m, {}).get("output", None)
    )
    metrics_df["cost_mean_per_M"]   = metrics_df["model_name"].map(
        lambda m: MODEL_COSTS.get(m, {}).get("mean", None)
    )
    return metrics_df


In [26]:

def compute_cost_efficiency(metrics_df: pd.DataFrame) -> pd.DataFrame:
    """
    Cost Efficiency Score = Action Accuracy / mean_cost.
    Нормируем mean_cost на минимальную стоимость среди моделей,
    чтобы получить безразмерный коэффициент.
    
    Чем выше — тем лучше (больше качества на рубль).
    """
    min_cost = metrics_df["cost_mean_per_M"].min()
    metrics_df["cost_efficiency"] = (
        metrics_df["action_accuracy"] / (metrics_df["cost_mean_per_M"] / min_cost)
    )
    return metrics_df

## метрики

In [27]:
# вспомогательные функции по иерархии

def luhmann_depth(luhmann_id: str) -> int:
    """
    Вычисляет глубину карточки по Luhmann-ID.
    
    Правила:
      "3"      → глубина 1 (корень)
      "3.1"    → глубина 2
      "3.1a"   → глубина 3
      "3.1a2"  → глубина 4
    
    Логика: считаем чередующиеся сегменты по токенам разбора.
    """
    if not luhmann_id or not isinstance(luhmann_id, str):
        return 0
    # Разбиваем на токены: числа, точки, буквы
    tokens = re.findall(r'\d+|[a-z]+', luhmann_id.lower())
    return len(tokens)


def luhmann_parent_prefix(luhmann_id: str) -> Optional[str]:
    """
    Возвращает префикс родителя для Luhmann-ID.
    
    "3.1a2" → "3.1a"
    "3.1a"  → "3.1"
    "3.1"   → "3"
    "3"     → None (корень)
    """
    if not luhmann_id or not isinstance(luhmann_id, str):
        return None
    tokens = re.findall(r'\d+|[a-z]+', luhmann_id.lower())
    if len(tokens) <= 1:
        return None
    # Восстанавливаем родительский ID из токенов без последнего
    parent_tokens = tokens[:-1]
    result = ""
    for i, tok in enumerate(parent_tokens):
        if i == 0:
            result = tok
        elif tok.isdigit() and parent_tokens[i-1].isalpha():
            result += tok
        elif tok.isalpha() and parent_tokens[i-1].isdigit():
            result += tok
        elif tok.isdigit() and parent_tokens[i-1].isdigit():
            result += f".{tok}"
        else:
            result += tok
    return result


def safe_kl_divergence(p: np.ndarray, q: np.ndarray, epsilon: float = 1e-9) -> float:
    """
    KL-дивергенция с защитой от нулей.
    P — эталон, Q — тест.
    """
    p = np.array(p, dtype=float) + epsilon
    q = np.array(q, dtype=float) + epsilon
    p = p / p.sum()
    q = q / q.sum()
    return float(np.sum(p * np.log(p / q)))

In [28]:
# функции с расчетом метрик
def metric_action_accuracy(ref: pd.DataFrame, test: pd.DataFrame) -> float:
    """
    Action Accuracy — доля карточек, где тестируемая модель выбрала
    то же действие (new_root / child_of / update_of), что и эталонная.
    
    Карточки сопоставляются по порядку (обе обрабатывают один и тот же
    список raw_cards в одном порядке).
    
    Returns: float в [0, 1], чем выше — тем лучше.
    """
    n = min(len(ref), len(test))
    ref_actions = ref["action"].iloc[:n].values
    test_actions = test["action"].iloc[:n].values
    matches = np.sum(ref_actions == test_actions)
    return float(matches / n) if n > 0 else 0.0


def metric_graph_depth_consistency(ref: pd.DataFrame, test: pd.DataFrame) -> float:
    """
    Graph Depth Consistency — насколько глубина встройки совпадает с эталоном.
    
    Считаем глубину каждого узла по его Luhmann-ID и сравниваем попарно.
    Нормируем на максимально возможное отклонение (max_depth эталона).
    
    Returns: float в [0, 1], чем выше — тем лучше.
    """
    n = min(len(ref), len(test))
    ref_depths = ref["luhmann_id"].iloc[:n].apply(luhmann_depth).values
    test_depths = test["luhmann_id"].iloc[:n].apply(luhmann_depth).values
    
    max_depth = max(ref_depths.max(), 1)
    mean_abs_diff = np.mean(np.abs(ref_depths - test_depths))
    
    # Нормируем: 0 отклонение → 1.0, отклонение на max_depth → 0.0
    return float(max(0.0, 1.0 - mean_abs_diff / max_depth))


def metric_root_rate_delta(ref: pd.DataFrame, test: pd.DataFrame) -> float:
    """
    Root Rate Delta — отклонение доли new_root-решений от эталона.
    
    Показывает, не "сплющивает" ли модель граф (делает всё корнями)
    или не "углубляет" ли чрезмерно (всё дочернее).
    
    Returns: float в [0, 1], чем НИЖЕ — тем лучше (0 = идеально).
    """
    ref_root_rate = (ref["action"] == "new_root").mean()
    test_root_rate = (test["action"] == "new_root").mean()
    return float(abs(ref_root_rate - test_root_rate))


def metric_child_attachment_precision(ref: pd.DataFrame, test: pd.DataFrame) -> float:
    """
    Child Attachment Precision — среди child_of решений, как часто
    тестируемая модель встраивает карточку на тот же уровень, что и эталон.
    
    Сравниваем родительский префикс Luhmann-ID (не точный UUID, а уровень).
    Это важно: даже если UUID родителя разный, уровень дерева должен совпадать.
    
    Returns: float в [0, 1], чем выше — тем лучше.
    """
    n = min(len(ref), len(test))
    ref_sub = ref.iloc[:n]
    test_sub = test.iloc[:n]
    
    # Рассматриваем только позиции, где ОБА выбрали child_of
    mask = (ref_sub["action"].values == "child_of") & (test_sub["action"].values == "child_of")
    
    if mask.sum() == 0:
        return 1.0  # нет child_of — нечего сравнивать, штрафа нет
    
    ref_depths = ref_sub["luhmann_id"].apply(luhmann_depth).values[mask]
    test_depths = test_sub["luhmann_id"].apply(luhmann_depth).values[mask]
    
    # Совпадение по глубине вложения = карточка встроена на правильный уровень
    level_match = np.sum(ref_depths == test_depths)
    return float(level_match / mask.sum())


def metric_action_distribution_kl(ref: pd.DataFrame, test: pd.DataFrame) -> float:
    """
    Action Distribution KL-Divergence — насколько распределение типов
    действий тестируемой модели отличается от эталонного.
    
    KL(P_ref ‖ Q_test), где P и Q — вероятности трёх классов действий.
    
    Returns: float >= 0, чем НИЖЕ — тем лучше (0 = идеальное совпадение).
    """
    actions = ["new_root", "child_of", "update_of"]
    
    ref_counts = ref["action"].value_counts()
    test_counts = test["action"].value_counts()
    
    p = np.array([ref_counts.get(a, 0) for a in actions], dtype=float)
    q = np.array([test_counts.get(a, 0) for a in actions], dtype=float)
    
    return safe_kl_divergence(p, q)


def metric_graph_structure_similarity(ref: pd.DataFrame, test: pd.DataFrame) -> float:
    """
    Graph Structure Similarity — корреляция Спирмена между распределением
    глубин узлов в тестовом и эталонном графах.
    
    Сравниваем гистограммы глубин (от 1 до max_depth) нормированные как PMF.
    Спирмен устойчив к выбросам и не требует нормальности.
    
    Returns: float в [-1, 1], чем выше — тем лучше (1 = идентичная структура).
    """
    ref_depths = ref["luhmann_id"].apply(luhmann_depth)
    test_depths = test["luhmann_id"].apply(luhmann_depth)
    
    max_depth = max(ref_depths.max(), test_depths.max(), 1)
    depth_range = list(range(1, max_depth + 1))
    
    # Строим гистограммы как нормированные распределения
    ref_hist = [( ref_depths == d).sum() for d in depth_range]
    test_hist = [(test_depths == d).sum() for d in depth_range]
    
    if sum(ref_hist) == 0 or sum(test_hist) == 0:
        return 0.0
    
    corr, _ = stats.spearmanr(ref_hist, test_hist)
    return float(corr) if not np.isnan(corr) else 0.0


def metric_latency_per_card(test: pd.DataFrame) -> float:
    """
    Latency per Card — среднее время обработки одной карточки (в секундах).
    
    latency_sec в DataFrame — суммарное время на весь прогон модели,
    поэтому делим на количество карточек.
    
    Returns: float > 0, чем НИЖЕ — тем лучше.
    """
    if "latency_sec" not in test.columns:
        return np.nan
    
    total_latency = test["latency_sec"].iloc[0]  # одно значение на весь прогон
    n = len(test)
    return float(total_latency / n) if n > 0 else np.nan


In [29]:
# агрегатор всех метрик
def compute_all_metrics(
    ref: pd.DataFrame,
    test: pd.DataFrame,
    model_name: str,
) -> Dict:
    """
    Вычисляет все метрики для одной тестируемой модели относительно эталона.
    
    Args:
        ref:        DataFrame эталонного прогона (reference)
        test:       DataFrame тестируемой модели
        model_name: строковое название модели (для отчёта)
    
    Returns:
        Словарь с названиями метрик и их значениями.
    """
    return {
        "model_name":                    model_name,
        "action_accuracy":               metric_action_accuracy(ref, test),
        "graph_depth_consistency":       metric_graph_depth_consistency(ref, test),
        "root_rate_delta":               metric_root_rate_delta(ref, test),        # ↓ лучше
        "child_attachment_precision":    metric_child_attachment_precision(ref, test),
        "graph_structure_similarity":    metric_graph_structure_similarity(ref, test),
        "action_kl_divergence":          metric_action_distribution_kl(ref, test), # ↓ лучше
        "latency_per_card_sec":          metric_latency_per_card(test),            # ↓ лучше
        "n_cards":                       min(len(ref), len(test)),
    }

In [30]:
def compute_row_by_row_metrics(ref: pd.DataFrame, test: pd.DataFrame, model_name: str) -> pd.DataFrame:
    """
    Создает детальный датафрейм со сравнением "карточка к карточке".
    Показывает точечно, где тестируемая модель ошиблась относительно эталона.
    """
    n = min(len(ref), len(test))
    ref_sub = ref.iloc[:n].reset_index(drop=True)
    test_sub = test.iloc[:n].reset_index(drop=True)
    
    rows = []
    for i in range(n):
        ref_row = ref_sub.iloc[i]
        test_row = test_sub.iloc[i]
        
        # 1. Совпадение действия
        action_match = bool(ref_row["action"] == test_row["action"])
        
        # 2. Разница глубин графа
        d_ref = luhmann_depth(ref_row.get("luhmann_id", ""))
        d_test = luhmann_depth(test_row.get("luhmann_id", ""))
        depth_diff = abs(d_ref - d_test)
        
        # 3. Совпадение уровня при child_of
        child_prefix_match = None
        if ref_row["action"] == "child_of" and test_row["action"] == "child_of":
            p_ref = luhmann_parent_prefix(ref_row.get("luhmann_id", ""))
            p_test = luhmann_parent_prefix(test_row.get("luhmann_id", ""))
            child_prefix_match = bool(p_ref == p_test)
            
        # Формируем строку детального отчета
        rows.append({
            "model_name": model_name,
            "zettel_id": ref_row.get("zettel_id"),
            "content_snippet": str(ref_row.get("content", ""))[:60] + "...",
            
            "action_match": action_match,
            "action_ref": ref_row.get("action"),
            "action_test": test_row.get("action"),
            
            "luhmann_ref": ref_row.get("luhmann_id"),
            "luhmann_test": test_row.get("luhmann_id"),
            "depth_diff": depth_diff,
            
            "child_prefix_match": child_prefix_match,
            
            "target_zettel_ref": ref_row.get("target_zettel_id"),
            "target_zettel_test": test_row.get("target_zettel_id"),
            
            "reasoning_ref": ref_row.get("reasoning"),
            "reasoning_test": test_row.get("reasoning")
        })
        
    return pd.DataFrame(rows)

##  запуск основного пайлайна с метриками

In [ ]:
def run_evaluation_pipeline():
    """
    Главная функция:
    1. Загружает эталон и тестовые файлы.
    2. Считает все метрики (Summary).
    3. Считает метрики по каждой карточке (Detailed).
    4. Сохраняет всё в Excel.
    """
    ref_df = load_reference(REFERENCE_FILE)
    test_dfs = load_test_files(LINKER_DIR)
    
    if not test_dfs:
        print(" Не найдено ни одного файла тестируемых моделей в директории.")
        return None
    
    all_metrics = []
    
    for model_key, test_df in test_dfs.items():
        print(f"\n -->Считаем метрики для: {model_key}")
        
        # === 1. РАСЧЕТ ИТОГОВЫХ МЕТРИК (САММАРИ) ===
        metrics = compute_all_metrics(ref=ref_df, test=test_df, model_name=model_key)
        all_metrics.append(metrics)
        
        # === 2. РАСЧЕТ ПОСТРОЧНЫХ МЕТРИК (ДЕТАЛЬНО) ===
        detailed_df = compute_row_by_row_metrics(ref=ref_df, test=test_df, model_name=model_key)
        
        # === 3. СОХРАНЕНИЕ ДЕТАЛЬНОГО ОТЧЕТА ===
        # Сохраняем построчный анализ для текущей модели
        detailed_file =  f"metric_results/linker/detailed_metrics_{model_key}.xlsx"
        detailed_df.to_excel(detailed_file, index=False)
        print(f"   [+] Сохранён детальный построчный отчёт: {detailed_file.name}")
        
        # Краткий вывод для консоли
        print(f"   Accuracy: {metrics['action_accuracy']:.3f} | Latency: {metrics['latency_per_card_sec']:.2f}s")
    
    # === 4. СОХРАНЕНИЕ ИТОГОВОГО САММАРИ ===
    metrics_df = pd.DataFrame(all_metrics)
    metrics_df = add_cost_columns(metrics_df)
    metrics_df = compute_cost_efficiency(metrics_df)
    metrics_df = metrics_df.sort_values("action_accuracy", ascending=False).reset_index(drop=True)
    
    metrics_df.to_excel(OUTPUT_FILE, index=False)
    print(f"\n✅ Итоговое саммари сохранено: {OUTPUT_FILE}")
    
    return metrics_df

In [35]:
if __name__ == "__main__":
    run_evaluation_pipeline()

[Reference] Загружено 484 карточек из synthetic_datasets/linker/reference_linker_gpt-5.2.xlsx
[Test] claude-haiku-4.5: 484 карточек из linker_anthropic_claude-haiku-4.5.xlsx
[Test] gemini-2.5-flash: 484 карточек из linker_google_gemini-2.5-flash.xlsx
[Test] gpt-4o-mini: 484 карточек из linker_openai_gpt-4o-mini.xlsx

 -->Считаем метрики для: claude-haiku-4.5
   [+] Сохранён детальный построчный отчёт: detailed_metrics_claude-haiku-4.5.xlsx
   Accuracy: 0.936 | Latency: 1.05s

 -->Считаем метрики для: gemini-2.5-flash
   [+] Сохранён детальный построчный отчёт: detailed_metrics_gemini-2.5-flash.xlsx
   Accuracy: 0.946 | Latency: 0.51s

 -->Считаем метрики для: gpt-4o-mini
   [+] Сохранён детальный построчный отчёт: detailed_metrics_gpt-4o-mini.xlsx
   Accuracy: 0.901 | Latency: 0.56s

✅ Итоговое саммари сохранено: synthetic_datasets/linker/linker_metrics_summary.xlsx
